In [ ]:
import os
import sys
import pandas as pd
import pickle as pkl
import numpy as np
import matplotlib.cm as cm
import plotly.io as pio
import torch
from scipy.stats import zscore
from sklearn.utils.extmath import randomized_svd

from pathlib import Path
project_root = Path(os.path.dirname(os.getcwd()))
sys.path.append(str(project_root.joinpath('src')))

from more_anns.models import AlexNet
from more_anns.losses import get_angle_representation

from utils.neuron_selectivity import save_average_resp_n_cc
from utils.data_loader import mouse_all_train_test_responses
from utils.helper import find_kn_unique_samples
from utils.plotting_functions import plot_points_w_errorbars, plot_hist_lines, \
    plot_models_overlay, model_to_color, MODEL_COLORS, \
    make_df_extractor, make_row_extractor, \
    plot_rotation_robustness, plot_orientation_robustness
from utils.robustness import load_imagenet_samples, evaluate_rotation_robustness, \
    evaluate_orientation_robustness, circular_error_ordered, DEFAULT_ANGLES

pio.kaleido.scope.mathjax = None

In [2]:
model_df = pd.read_csv( project_root / 'params' / 'model_params.csv', header=0)
params_df = pd.read_csv( project_root / 'params' / 'fixed_params.csv', header=0, index_col=0)

n_neurons = int(params_df.loc['n_neurons'].value)
d = int(params_df.loc['n_procs'].value)
results_folder = project_root / 'results_extended_anns'
results_folder.mkdir(parents=True, exist_ok=True)

In [3]:
C0 = 0.001
equivariant = False
model = False

if equivariant:
    suffix = f'_{d}dim'
    if model:
        suffix += '_model'
else:
    suffix = ''

color_pts='rgba(140, 140, 140, 1)' 
color_err='rgba(169, 169, 169, 1.0)'

In [ ]:
COLORS = {
    'alexnet':                 'rgba(100, 100, 100, 1)',
    'random_alexnet':          'rgba(180, 180, 180, 1)',
    'alt_alexnet_rot-aug':     'rgba(44, 160, 44, 1)',
    'alt_alexnet_ori':         'rgba(227, 119, 194, 1)',
    'alt_alexnet_multi-task':  'rgba(214, 39, 40, 1)',
}
MODEL_COLORS.update(COLORS)

models = model_df['network'].unique()
layers = model_df[model_df['network']==models[0]]['layer'].values
print(f'Models: {list(models)}')
print(f'Layers: {list(layers)}')

### plot decoding errors

In [24]:
svm_df = pd.DataFrame()
for i0 in range(model_df.shape[0]):
    model_name = model_df['network'].values[i0]
    layer_name = model_df['layer'].values[i0]
    cache_folder = project_root / f'cache' / model_name

    class_f = cache_folder / f'{layer_name}_class2x_decoder_{n_neurons}avg_{C0}C{suffix}.csv'
    if not class_f.exists():
        print(f'No classification data named {class_f}, skipping.')
        continue
    tmp_df = pd.read_csv(class_f)
    tmp_df['model'] = model_name
    tmp_df['area'] = layer_name
    tmp_df['name'] = f'{model_name}_{layer_name}'
    svm_df = pd.concat([svm_df, tmp_df], axis=0)

name_to_x_dict = {name: i for i, name in enumerate(svm_df['name'].unique())}

svm_plot_df = svm_df.groupby(['name','model'])['acc test'].mean().reset_index()
svm_plot_df['y'] = 1-svm_plot_df['acc test']
svm_plot_df['x'] = [name_to_x_dict[name] for name in svm_plot_df['name']]
svm_plot_df['xtick'] = [name_to_x_dict[name] for name in svm_plot_df['name']]
svm_plot_df.sort_values('x', inplace=True)
svm_plot_df['errx'] = 0
svm_plot_df['erry'] = svm_df.groupby(['name','model'])['acc test'].std().values
svm_plot_df['erry'] = svm_plot_df['erry']/np.sqrt(svm_df.groupby(['name','model'])['acc test'].count().values)
svm_plot_df['color'] = color_pts
svm_plot_df['errcolor'] = color_err

In [25]:
fig = plot_points_w_errorbars(svm_plot_df, y_dp=[1-1/91], sz=8, height=150, width=680, alpha=.6)
for i in range(model_df.shape[0] // 5 + 1):
    fig.add_vline(x=i*5-0.5, line_width=1, line_dash='dash', line_color='rgba(200, 200, 200, 0.8)')
fig.update_layout(yaxis_title='decoding error (1)')
fig.update_xaxes(title_standoff=0, tickvals=svm_plot_df['xtick'].unique(), ticktext=[s[-5:] for s in svm_plot_df['name'].unique()], tickangle=-45)
fig.update_yaxes(title_standoff=0, range=[-0.03,1.03], tickvals=[0,0.25,0.5,0.75,1], ticktext=['0','0.25','0.5','0.75','1'])
fig.write_image(results_folder / f'more_anns_{C0}C_{n_neurons}avg{suffix}.pdf', scale=1, engine='kaleido')
fig.show()

In [ ]:
# Decoding error — models overlaid across layers
get_decoding_err = make_df_extractor(svm_df, 'model', 'area', 'acc test', transform=lambda x: 1-x)

fig = plot_models_overlay(models, layers, get_decoding_err,
                          ylabel='decoding error (1)', yrange=[0.45, 1.03],
                          y_dp=[1-1/91],
                          save_path=results_folder / f'more_anns_decoding_error_by_model{suffix}.pdf')

In [27]:
results_df = pd.DataFrame()
for i0 in range(model_df.shape[0]):
    model_name = model_df['network'].values[i0]
    layer_name = model_df['layer'].values[i0]
    cache_folder = project_root / f'cache' / model_name

    ori_f = cache_folder / f'{layer_name}_ori2x_decoder_0ref_{n_neurons}avg{suffix}.csv'
    if not ori_f.exists():
        print(f'No orientation data named {ori_f}, skipping.')
        continue
    tmp_df = pd.read_csv(ori_f)
    tmp_df['model'] = model_name
    tmp_df['area'] = layer_name
    tmp_df['name'] = f'{model_name}_{layer_name}'
    results_df = pd.concat([results_df, tmp_df], axis=0)

name_to_x_dict = {name: i for i, name in enumerate(results_df['name'].unique())}

#compute mean and sem for each area
plot_df = results_df.groupby(['name'])['res'].mean().reset_index()
plot_df['y'] = plot_df['res']
plot_df['x'] = [name_to_x_dict[name] for name in plot_df['name']]
plot_df['xtick'] = [name_to_x_dict[name] for name in plot_df['name']]
plot_df.sort_values('x', inplace=True)
plot_df['errx'] = 0
plot_df['erry'] = results_df.groupby(['name'])['res'].std().reset_index()['res']
plot_df['erry'] = plot_df['erry']/np.sqrt(results_df.groupby(['name'])['res'].count().values)
plot_df['color'] = color_pts
plot_df['errcolor'] = color_err

In [28]:
fig = plot_points_w_errorbars(plot_df, y_dp=[90], sz=8, height=150, width=680, alpha=.6)
for i in range(model_df.shape[0] // 5 + 1):
    fig.add_vline(x=i*5-0.5, line_width=1, line_dash='dash', line_color='rgba(200, 200, 200, 0.8)')
fig.update_layout(yaxis_title='absolute angle estimation error (deg)')
fig.update_xaxes(title_standoff=0, tickvals=svm_plot_df['xtick'].unique(), ticktext=[s[-5:] for s in svm_plot_df['name'].unique()], tickangle=-45)
fig.update_yaxes(title_standoff=0, range=[0,100], tickvals=[0,45,90,180], ticktext=['0 ','45 ','90 ','180 '])
fig.write_image(results_folder / f'more_anns_ori_0ref_{n_neurons}avg{suffix}.pdf', scale=1, engine='kaleido')
fig.show()

In [ ]:
# Orientation error — models overlaid across layers
get_orient_err = make_df_extractor(results_df, 'model', 'area', 'res')

fig = plot_models_overlay(models, layers, get_orient_err,
                          ylabel='absolute angle estimation error (deg)', yrange=[0, 100],
                          y_dp=[90],
                          save_path=results_folder / f'more_anns_orient_error_by_model{suffix}.pdf')

### plot equivariance summary

In [30]:
color_inv='rgba(247, 148, 29, 1)'
color_tuned='rgba(139, 94, 60, 1)'

#collect dim results across layers
dim_df = pd.DataFrame()
for i0 in range(model_df.shape[0]):
    model_name = model_df['network'].values[i0]
    layer_name = model_df['layer'].values[i0]
    cache_folder = project_root / f'cache' / model_name

    var_f = cache_folder / f'{layer_name}_var_dim_{n_neurons}avg_{d}dim.csv'
    if not var_f.exists():
        continue
    tmp_df = pd.read_csv(var_f)
    tmp_df['dim'] = 0.0
    tmp_df['erry'] = 0.0
    tmp_df.loc[0,'dim'] = tmp_df.groupby('stim')['evals'].apply(lambda x: (np.abs(x)<0.5).sum()).mean()
    tmp_df.loc[0,'erry'] = tmp_df.groupby('stim')['evals'].apply(lambda x: (np.abs(x)<0.5).sum()).std()
    tmp_df.loc[0,'erry'] = tmp_df.loc[0,'erry']/np.sqrt(tmp_df['stim'].nunique())
    tmp_df.loc[1,'dim'] = tmp_df.groupby('stim')['evals'].apply(lambda x: ((np.abs(x)>=0.5)&(np.abs(x)<1.5)).sum()).mean()
    tmp_df.loc[1,'erry'] = tmp_df.groupby('stim')['evals'].apply(lambda x: ((np.abs(x)>=0.5)&(np.abs(x)<1.5)).sum()).std()
    tmp_df.loc[1,'erry'] = tmp_df.loc[1,'erry']/np.sqrt(tmp_df['stim'].nunique())
    tmp_df['model'] = model_name
    tmp_df['area'] = layer_name
    tmp_df['name'] = f'{model_name}_{layer_name}'
    dim_df = pd.concat([dim_df, tmp_df.iloc[:2].reset_index(names='cat')], axis=0)

name_to_x_dict = {name: i for i, name in enumerate(dim_df['name'].unique())}

#find means and sems grouped by name
plot_df = dim_df.groupby(['name','cat'])['dim'].mean().reset_index()
plot_df['erry'] = dim_df.groupby(['name','cat'])['erry'].apply(lambda x: (x**2).sum()**0.5).values
plot_df['erry'] = plot_df['erry']/np.sqrt(dim_df.groupby(['name','cat'])['dim'].count().values)
plot_df['erry'] = plot_df['erry'].replace(np.nan, 0)

color_to_dx_dict = {f'{color_inv}': -0.1, f'{color_tuned}': 0.1}
plot_df['color'] = color_inv
idx_tuned = plot_df[plot_df['cat']==1].index
plot_df.loc[idx_tuned,'color'] = color_tuned
plot_df['errcolor'] = color_inv
plot_df.loc[idx_tuned,'errcolor'] = color_tuned
plot_df['y'] = plot_df['dim']
plot_df['x'] = [name_to_x_dict[name] + color_to_dx_dict[color] for name, color in zip(plot_df['name'], plot_df['color'])]
plot_df['xtick'] = [name_to_x_dict[name] for name in plot_df['name']]
plot_df.sort_values('x', inplace=True)
plot_df['errx'] = 0

In [31]:
fig = plot_points_w_errorbars(plot_df, sz=8, height=150, width=680, alpha=.6)
for i in range(model_df.shape[0] // 5 + 1):
    fig.add_vline(x=i*5-0.5, line_width=1, line_dash='dash', line_color='rgba(200, 200, 200, 0.8)')
fig.update_layout(yaxis_title='dimension (1)')
fig.update_xaxes(title_standoff=0, tickvals=plot_df['xtick'].unique(), ticktext=[s[-5:] for s in plot_df['name'].unique()], tickangle=-45)
fig.update_yaxes(title_standoff=0, range=[0,20], tickvals=[0,10,20], ticktext=['0','10','20'])
fig.write_image(results_folder / f'more_anns_dim_{n_neurons}avg_{d}dim.pdf', scale=1, engine='kaleido')
fig.show()

In [ ]:
# Invariant dimension — models overlaid across layers
get_inv_dim = make_row_extractor(dim_df, 'model', 'cat', 0, 'dim', 'erry', layer_col='area')

fig = plot_models_overlay(models, layers, get_inv_dim,
                          ylabel='invariant dimension (1)', yrange=[0, 20],
                          save_path=results_folder / f'more_anns_inv_dim_by_model.pdf')

In [ ]:
# Tuned dimension — models overlaid across layers
get_tuned_dim = make_row_extractor(dim_df, 'model', 'cat', 1, 'dim', 'erry', layer_col='area')

fig = plot_models_overlay(models, layers, get_tuned_dim,
                          ylabel='tuned dimension (1)', yrange=[0, 20],
                          save_path=results_folder / f'more_anns_tuned_dim_by_model.pdf')

In [34]:
#collect pcorr results across layers
var_df = pd.DataFrame()
for i0 in range(model_df.shape[0]):
    model_name = model_df['network'].values[i0]
    layer_name = model_df['layer'].values[i0]
    cache_folder = project_root / f'cache' / model_name

    var_f = cache_folder / f'{layer_name}_var_dim_{n_neurons}avg_{d}dim.csv'
    if not var_f.exists():
        continue
    tmp_df = pd.read_csv(var_f)
    tmp_df['model'] = model_name
    tmp_df['area'] = layer_name
    tmp_df['name'] = f'{model_name}_{layer_name}'
    var_df = pd.concat([var_df, tmp_df.copy()], axis=0)
var_df = var_df.reset_index(drop=True)

#invariant dimensions
inv_all_df = var_df[np.abs(var_df['evals'])<0.5]
inv_pc_df = inv_all_df.groupby(['name','model','area','stim'])['pc test'].mean().reset_index()

#tuned dimensions
tuned_all_df = var_df[(np.abs(var_df['evals'])>=0.5)&(np.abs(var_df['evals'])<1.5)]
tuned_pc_df = tuned_all_df.groupby(['name','model','area','stim'])['pc test'].mean().reset_index()

#compute means and sems grouped by name
inv_pc_avg_df = inv_pc_df.groupby(['name','model','area'])['pc test'].mean().reset_index().rename(columns={'pc test': 'pc'})
inv_pc_avg_df['erry'] = inv_pc_df.groupby(['name','model','area'])['pc test'].std().values/np.sqrt(var_df['stim'].nunique())
inv_pc_avg_df['cat'] = 0
tuned_pc_avg_df = tuned_pc_df.groupby(['name','model','area'])['pc test'].mean().reset_index().rename(columns={'pc test': 'pc'})
tuned_pc_avg_df['erry'] = tuned_pc_df.groupby(['name','model','area'])['pc test'].std().values/np.sqrt(var_df['stim'].nunique())
tuned_pc_avg_df['cat'] = 1
pc_df = pd.concat([inv_pc_avg_df, tuned_pc_avg_df], axis=0).reset_index(drop=True)

name_to_x_dict = {name: i for i, name in enumerate(pc_df['name'].unique())}
color_to_dx_dict = {f'{color_inv}': -0.1, f'{color_tuned}': 0.1}

plot_df = pc_df.groupby(['name','cat'])['pc'].mean().reset_index()
plot_df['color'] = color_inv
idx_tuned = plot_df[plot_df['cat']==1].index
plot_df.loc[idx_tuned,'color'] = color_tuned
plot_df['errcolor'] = color_inv
plot_df.loc[idx_tuned,'errcolor'] = color_tuned
plot_df['y'] = plot_df['pc']
plot_df['erry'] = pc_df.groupby(['name','cat'])['erry'].apply(lambda x: (x**2).sum()**0.5).values
plot_df['erry'] = plot_df['erry']/np.sqrt(pc_df.groupby(['name','cat'])['pc'].count().values)
plot_df['erry'] = plot_df['erry'].replace(np.nan, 0)
plot_df['x'] = [name_to_x_dict[name] + color_to_dx_dict[color] for name, color in zip(plot_df['name'], plot_df['color'])]
plot_df['xtick'] = [name_to_x_dict[name] for name in plot_df['name']]
plot_df.sort_values('x', inplace=True)
plot_df['errx'] = 0

In [35]:
fig = plot_points_w_errorbars(plot_df, y_dp=None, sz=8, height=150, width=680, alpha=.6)
for i in range(model_df.shape[0] // 5 + 1):
    fig.add_vline(x=i*5-0.5, line_width=1, line_dash='dash', line_color='rgba(200, 200, 200, 0.8)')
fig.update_layout(yaxis_title='Pearson correlation (1)')
fig.update_xaxes(title_standoff=0, tickvals=plot_df['xtick'].unique(), ticktext=[s[-5:] for s in plot_df['name'].unique()], tickangle=-45)
fig.update_yaxes(title_standoff=0, range=[0,1])
fig.write_image(results_folder / f'more_anns_pcorr_{n_neurons}avg_{d}dim.pdf', scale=1, engine='kaleido')
fig.show()

In [ ]:
# Invariant Pearson correlation — models overlaid across layers
get_inv_pcorr = make_row_extractor(pc_df, 'name', 'cat', 0, 'pc', 'erry')

fig = plot_models_overlay(models, layers, get_inv_pcorr,
                          ylabel='invariant Pearson correlation (1)', yrange=[0, 1],
                          save_path=results_folder / f'more_anns_inv_pcorr_by_model.pdf')

In [ ]:
# Tuned Pearson correlation — models overlaid across layers
get_tuned_pcorr = make_row_extractor(pc_df, 'name', 'cat', 1, 'pc', 'erry')

fig = plot_models_overlay(models, layers, get_tuned_pcorr,
                          ylabel='tuned Pearson correlation (1)', yrange=[0, 1],
                          save_path=results_folder / f'more_anns_tuned_pcorr_by_model.pdf')

### plot equivariance quantification

In [38]:
#collect ccnorm results across layers
performance_df = pd.DataFrame()
for i0 in range(model_df.shape[0]):
    model_name = model_df['network'].values[i0]
    layer_name = model_df['layer'].values[i0]
    cache_folder = project_root / f'cache' / model_name

    equi_f = cache_folder / f'{layer_name}_equivariance_{n_neurons}avg_{d}dim.csv'
    if not equi_f.exists():
        continue
    tmp_df = pd.read_csv(equi_f)
    tmp_df['model'] = model_name
    tmp_df['area'] = layer_name
    tmp_df['name'] = f'{model_name}_{layer_name}'
    performance_df = pd.concat([performance_df, tmp_df], axis=0)

name_to_x_dict = {name: i for i, name in enumerate(performance_df['name'].unique())}

#compute mean and sem for each name
plot_df = performance_df.groupby(['name'])[['cc_mean']].mean().reset_index()
plot_df['y'] = plot_df['cc_mean']
plot_df['x'] = [name_to_x_dict[name] for name in plot_df['name']]
plot_df['xtick'] = [name_to_x_dict[name] for name in plot_df['name']]
plot_df.sort_values('x', inplace=True)
plot_df['errx'] = 0
plot_df['erry'] = performance_df.groupby(['name'])['cc_mean'].std().values
plot_df['erry'] = plot_df['erry']/np.sqrt(performance_df.groupby(['name'])['cc_mean'].count().values)
plot_df['color'] = color_pts
plot_df['errcolor'] = color_err

In [39]:
fig = plot_points_w_errorbars(plot_df, y_dp=[0], sz=8, height=150, width=680, alpha=.6)
for i in range(model_df.shape[0] // 5 + 1):
    fig.add_vline(x=i*5-0.5, line_width=1, line_dash='dash', line_color='rgba(200, 200, 200, 0.8)')
fig.update_layout(yaxis_title='CCnorm (1)')
fig.update_xaxes(title_standoff=0, tickvals=plot_df['xtick'].unique(), ticktext=[s[-5:] for s in plot_df['name'].unique()], tickangle=-45)
fig.update_yaxes(title_standoff=0, range=[-0.05, 0.85], tickvals=[0,0.2,0.4,0.6,0.8], ticktext=['0','0.2','0.4','0.6','0.8'])
fig.write_image(results_folder / f'more_anns_cc_{n_neurons}avg_{d}dim.pdf', scale=1, engine='kaleido')
fig.show()

In [ ]:
# CCnorm — models overlaid across layers
get_ccnorm = make_df_extractor(performance_df, 'model', 'area', 'cc_mean')

fig = plot_models_overlay(models, layers, get_ccnorm,
                          ylabel='CCnorm (1)', yrange=[-0.05, 1.05],
                          y_dp=[0],
                          save_path=results_folder / f'more_anns_ccnorm_by_model.pdf')

In [41]:
#collect variance results across layers
var_df = pd.DataFrame()
for i0 in range(model_df.shape[0]):
    model_name = model_df['network'].values[i0]
    layer_name = model_df['layer'].values[i0]
    cache_folder = project_root / f'cache' / model_name

    var_f = cache_folder / f'{layer_name}_var_dim_{n_neurons}avg_{d}dim.csv'
    if not var_f.exists():
        continue
    tmp_df = pd.read_csv(var_f)
    tmp_df['model'] = model_name
    tmp_df['area'] = layer_name
    tmp_df['name'] = f'{model_name}_{layer_name}'
    var_df = pd.concat([var_df, tmp_df.copy()], axis=0)
var_df = var_df.reset_index(drop=True)

#invariant dimensions
inv_all_df = var_df[np.abs(var_df['evals'])<0.5]
inv_pc_df = inv_all_df.groupby(['name','model','area','stim'])['var test'].sum().reset_index()

#tuned dimensions
tuned_all_df = var_df[(np.abs(var_df['evals'])>=0.5)&(np.abs(var_df['evals'])<1.5)]
tuned_pc_df = tuned_all_df.groupby(['name','model','area','stim'])['var test'].sum().reset_index()

#compute means and sems grouped by name
inv_pc_avg_df = inv_pc_df.groupby(['name','model','area'])['var test'].mean().reset_index().rename(columns={'var test': 'var'})
inv_pc_avg_df['erry'] = inv_pc_df.groupby(['name','model','area'])['var test'].std().values/np.sqrt(var_df['stim'].nunique())
inv_pc_avg_df['cat'] = 0
tuned_pc_avg_df = tuned_pc_df.groupby(['name','model','area'])['var test'].mean().reset_index().rename(columns={'var test': 'var'})
tuned_pc_avg_df['erry'] = tuned_pc_df.groupby(['name','model','area'])['var test'].std().values/np.sqrt(var_df['stim'].nunique())
tuned_pc_avg_df['cat'] = 1
pc_df = pd.concat([inv_pc_avg_df, tuned_pc_avg_df], axis=0).reset_index(drop=True)

name_to_x_dict = {name: i for i, name in enumerate(pc_df['name'].unique())}
color_to_dx_dict = {f'{color_inv}': -0.1, f'{color_tuned}': 0.1}

plot_df = pc_df.groupby(['name','cat'])['var'].mean().reset_index()
plot_df['color'] = color_inv
idx_tuned = plot_df[plot_df['cat']==1].index
plot_df.loc[idx_tuned,'color'] = color_tuned
plot_df['errcolor'] = color_inv
plot_df.loc[idx_tuned,'errcolor'] = color_tuned
plot_df['y'] = plot_df['var']
plot_df['erry'] = pc_df.groupby(['name','cat'])['erry'].apply(lambda x: (x**2).sum()**0.5).values
plot_df['erry'] = plot_df['erry']/np.sqrt(pc_df.groupby(['name','cat'])['var'].count().values)
plot_df['erry'] = plot_df['erry'].replace(np.nan, 0)
plot_df['x'] = [name_to_x_dict[name] + color_to_dx_dict[color] for name, color in zip(plot_df['name'], plot_df['color'])]
plot_df['xtick'] = [name_to_x_dict[name] for name in plot_df['name']]
plot_df.sort_values('x', inplace=True)
plot_df['errx'] = 0

In [42]:
fig = plot_points_w_errorbars(plot_df, y_dp=None, sz=8, height=150, width=680, alpha=.6)
for i in range(model_df.shape[0] // 5 + 1):
    fig.add_vline(x=i*5-0.5, line_width=1, line_dash='dash', line_color='rgba(200, 200, 200, 0.8)')
fig.update_layout(yaxis_title='Variance explained (1)')
fig.update_xaxes(title_standoff=0, tickvals=plot_df['xtick'].unique(), ticktext=[s[-5:] for s in plot_df['name'].unique()], tickangle=-45)
fig.update_yaxes(title_standoff=0, range=[0,0.2], tickvals=[0,0.1,0.2], ticktext=['0','10','20'])
fig.write_image(results_folder / f'more_anns_var_{n_neurons}avg_{d}dim.pdf', scale=1, engine='kaleido')
fig.show()

In [ ]:
# Invariant variance — models overlaid across layers
get_inv_var = make_row_extractor(pc_df, 'name', 'cat', 0, 'var', 'erry', scale=100.0)

fig = plot_models_overlay(models, layers, get_inv_var,
                          ylabel='invariant variance explained (1)', yrange=[0, 60],
                          save_path=results_folder / f'more_anns_inv_var_by_model.pdf')

In [ ]:
# Tuned variance — models overlaid across layers
get_tuned_var = make_row_extractor(pc_df, 'name', 'cat', 1, 'var', 'erry', scale=100.0)

fig = plot_models_overlay(models, layers, get_tuned_var,
                          ylabel='tuned variance explained (1)', yrange=[0, 60],
                          save_path=results_folder / f'more_anns_tuned_var_by_model.pdf')

### rotation robustness

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ANGLES = DEFAULT_ANGLES
cache_file = project_root / 'cache' / 'imagenet_val_100_seed42.pkl'
samples = load_imagenet_samples(cache_file)

# Pretrained AlexNet (ImageNet)
model_pretrained = AlexNet(task='classification', pretrained_weights='imagenet').to(DEVICE)
model_pretrained.eval()
correct_pretrained = evaluate_rotation_robustness(model_pretrained, samples, ANGLES, DEVICE)

# Random AlexNet
model_random = AlexNet(task='classification', pretrained_weights=None).to(DEVICE)
model_random.eval()
correct_random = evaluate_rotation_robustness(model_random, samples, ANGLES, DEVICE)

# rot-aug: classification-only with rotation augmentation
ckpt_rot_aug = torch.load(project_root / 'cache' / 'more_anns' / 'rot-aug' / 'best_model.pth',
                           map_location='cpu', weights_only=True)
model_rot_aug = AlexNet(task='classification', pretrained_weights=None)
model_rot_aug.load_state_dict(ckpt_rot_aug['model_state_dict'])
model_rot_aug = model_rot_aug.to(DEVICE)
model_rot_aug.eval()
correct_rot_aug = evaluate_rotation_robustness(model_rot_aug, samples, ANGLES, DEVICE)

# multi-task: classification + ordered orientation
ckpt_mt = torch.load(project_root / 'cache' / 'more_anns' / 'multi-task' / 'best_model.pth',
                      map_location='cpu', weights_only=True)
cfg_mt = ckpt_mt['config']
model_mt = AlexNet(
    task='multitask', similarity_layer=cfg_mt['similarity_layer'],
    orientation_fusion=cfg_mt['orientation_fusion'],
    orientation_output_dim=2,
    orientation_activation=cfg_mt.get('orientation_activation', 'linear'),
)
model_mt.load_state_dict(ckpt_mt['model_state_dict'])
model_mt = model_mt.to(DEVICE)
model_mt.eval()
correct_mt = evaluate_rotation_robustness(model_mt, samples, ANGLES, DEVICE, use_backbone=True)

In [ ]:
rotation_results = {
    'random': correct_random,
    'pretrained': correct_pretrained,
    'rot-aug': correct_rot_aug,
    'multi-task': correct_mt,
}
fig = plot_rotation_robustness(rotation_results, ANGLES, project_root, results_folder=results_folder)
fig.show()

### orientation robustness

In [ ]:
rep_ordered = get_angle_representation('sincos_ordered')

# ori: orientation-only (sincos_ordered)
ckpt_ori = torch.load(project_root / 'cache' / 'more_anns' / 'ori' / 'best_model.pth',
                       map_location='cpu', weights_only=True)
cfg_ori = ckpt_ori['config']
model_ori = AlexNet(
    task='orientation', similarity_layer=cfg_ori['similarity_layer'],
    orientation_fusion=cfg_ori['orientation_fusion'],
    orientation_output_dim=2,
    orientation_activation=cfg_ori.get('orientation_activation', 'linear'),
)
model_ori.load_state_dict(ckpt_ori['model_state_dict'])
model_ori = model_ori.to(DEVICE)
model_ori.eval()

# Random baseline: same architecture as ori, random weights
model_orient_random = AlexNet(
    task='orientation', similarity_layer=cfg_ori['similarity_layer'],
    orientation_fusion=cfg_ori['orientation_fusion'],
    orientation_output_dim=2,
    orientation_activation=cfg_ori.get('orientation_activation', 'linear'),
).to(DEVICE)
model_orient_random.eval()

pred_angles_ori = evaluate_orientation_robustness(model_ori, samples, rep_ordered, ANGLES, DEVICE, mode='orientation')
pred_angles_mt = evaluate_orientation_robustness(model_mt, samples, rep_ordered, ANGLES, DEVICE, mode='multitask')
pred_angles_random = evaluate_orientation_robustness(model_orient_random, samples, rep_ordered, ANGLES, DEVICE, mode='orientation')

In [ ]:
true_diff = ANGLES.astype(float)
errors_ori = circular_error_ordered(pred_angles_ori, true_diff[None, :])
errors_mt = circular_error_ordered(pred_angles_mt, true_diff[None, :])
errors_random = circular_error_ordered(pred_angles_random, true_diff[None, :])

orientation_results = {
    'random': errors_random,
    'ori': errors_ori,
    'multi-task': errors_mt,
}
fig = plot_orientation_robustness(orientation_results, ANGLES, project_root, results_folder=results_folder)
fig.show()